# ResNet-50 Model Upload Pipeline: A Developer Template

[![Model](https://img.shields.io/badge/Model-ResNet--50-blue.svg)](https://arxiv.org/abs/1512.03385)
[![Dataset](https://img.shields.io/badge/Dataset-Stanford%20Cars-green.svg)](https://ai.stanford.edu/~jkrause/cars/car_dataset.html)
[![Accuracy](https://img.shields.io/badge/Accuracy-87.3%25-brightgreen.svg)]()
[![Inference](https://img.shields.io/badge/Inference-45ms-orange.svg)]()

## Overview

This notebook provides a comprehensive, reusable template for uploading a computer vision model to the Zededa EdgeAI platform. It uses a **ResNet-50** model, fine-tuned on the Stanford Cars dataset, as a working example.

The primary goal of this notebook is to serve as a foundation for developers. You can adapt the steps outlined here to upload your own models by changing a few key variables.

### Example Model Specifications
- **Architecture**: ResNet-50 (Deep Residual Network)
- **Task**: Fine-grained vehicle classification
- **Dataset**: Stanford Cars (196 vehicle categories)
- **Format**: ONNX (optimized for edge deployment)
- **Input**: 224×224 RGB images
- **Framework**: PyTorch → ONNX conversion

### Pipeline Steps
1. **Configuration**: Set key variables for your model.
2. **Environment Setup**: Clear and verify authentication credentials.
3. **Dependency Installation**: Install required Python packages.
4. **Model Loading & Analysis**: Load the ONNX model and perform deep analysis.
5. **Performance Evaluation**: (Optional) Evaluate model performance against a test dataset.
6. **Configuration Generation**: Automatically create Triton `config.pbtxt` and other artifacts.
7. **Model Upload**: Use MLflow to upload the model and its metadata to the registry.
8. **Registration & Deployment**: Transition the model to the "Production" stage.
9. **Verification**: Confirm the upload was successful.

In [1]:
# Package installation and verification
import subprocess
import sys
import pkg_resources
from packaging import version

def check_package_installed(package_name, min_version=None):
    """Check if a package is installed and optionally meets minimum version requirement."""
    try:
        installed_version = pkg_resources.get_distribution(package_name).version
        if min_version and version.parse(installed_version) < version.parse(min_version):
            return False, installed_version
        return True, installed_version
    except pkg_resources.DistributionNotFound:
        return False, None

def install_package(package_spec):
    """Install a package using pip."""
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", package_spec])
        return True
    except subprocess.CalledProcessError:
        return False

# Core required packages
required_packages = {
    "numpy": "2.0.0", "pandas": "2.3.0", "scikit-learn": "1.6.0", "scipy": "1.13.0",
    "torch": "2.8.0", "torchvision": "0.23.0", "onnx": "1.19.0", "onnxruntime": "1.19.0",
    "onnx2torch": "1.5.15", "torchinfo": "1.8.0", "thop": "0.1.1",
    "mlflow": "3.1.4", "boto3": "1.40.0", "requests": "2.32.0", "zededa-edgeai-sdk": "1.0.4",
    "pillow": "11.3.0", "huggingface-hub": "0.35.0", "wget": "3.2", "tqdm": "4.67.0"
}

print("🔍 Checking packages...")
missing_packages = []
for package, min_ver in required_packages.items():
    is_installed, current_ver = check_package_installed(package, min_ver)
    if not is_installed:
        if current_ver is None:
            missing_packages.append(f"{package}>={min_ver}")
        else:
            missing_packages.append(f"{package}>={min_ver}")

if missing_packages:
    print(f"📦 Installing {len(missing_packages)} packages...")
    failed_installs = []
    for package_spec in missing_packages:
        if not install_package(package_spec):
            failed_installs.append(package_spec)
    
    if failed_installs:
        print(f"⚠️  Failed to install: {', '.join(failed_installs)}")
else:
    print("✅ All packages up-to-date")

# Verify critical imports
critical_imports = ["numpy", "torch", "onnx", "onnxruntime", "mlflow", "PIL"]
import_failures = []
for module in critical_imports:
    try:
        __import__(module)
    except ImportError:
        import_failures.append(module)

if import_failures:
    print(f"❌ Import failures: {', '.join(import_failures)}")
else:
    print("✅ All critical imports verified")

🔍 Checking packages...
✅ All packages up-to-date
✅ All critical imports verified
✅ All critical imports verified


## Step 1: Configuration

> **Developer Note:** This is the main section you need to modify for your own model. Update the variables below to match your model's file path, name, and the dataset it was trained on.

In [2]:
# Model download
import os
from huggingface_hub import hf_hub_download
import shutil

repo_id = "zededa/resnet50-cars"
filename = "resnet50_cars_enhanced.onnx"

if os.path.exists(filename):
    local_model_path = os.path.abspath(filename)
    print(f"✅ Model exists: {filename} ({os.path.getsize(filename)/1024/1024:.1f} MB)")
else:
    print(f"📥 Downloading model from {repo_id}...")
    try:
        cached_path = hf_hub_download(repo_id=repo_id, filename=filename)
        local_model_path = os.path.abspath(filename)
        shutil.copy2(cached_path, local_model_path)
        print(f"✅ Downloaded: {filename} ({os.path.getsize(filename)/1024/1024:.1f} MB)")
    except Exception as e:
        print(f"❌ Download failed: {e}")
        local_model_path = None

✅ Model exists: resnet50_cars_enhanced.onnx (94.0 MB)


In [3]:
# Dataset download
import wget

url = "https://huggingface.co/datasets/zededa/stanford-cars/resolve/main/stanford-cars.zip"
zip_file = "stanford-cars.zip"
dataset_folder = "stanford-cars"

if os.path.exists(dataset_folder):
    test_classes = len([d for d in os.listdir(f"{dataset_folder}/test") if os.path.isdir(f"{dataset_folder}/test/{d}")]) if os.path.exists(f"{dataset_folder}/test") else 0
    print(f"✅ Dataset exists: {test_classes} test classes")
elif os.path.exists(zip_file):
    print(f"✅ ZIP exists: {zip_file} ({os.path.getsize(zip_file)/1024/1024:.1f} MB)")
else:
    print(f"📥 Downloading dataset...")
    try:
        wget.download(url, out=".")
        print(f"\n✅ Downloaded: {zip_file}")
    except Exception as e:
        print(f"❌ Download failed: {e}")

✅ Dataset exists: 196 test classes


In [4]:
# Dataset extraction
import zipfile

if os.path.exists("stanford-cars"):
    print("✅ Dataset already extracted")
elif os.path.exists("stanford-cars.zip"):
    print("📦 Extracting dataset...")
    try:
        with zipfile.ZipFile("stanford-cars.zip", 'r') as zip_ref:
            zip_ref.extractall('.')
        print("✅ Extraction complete")
    except Exception as e:
        print(f"❌ Extraction failed: {e}")
else:
    print("❌ No ZIP file found")

✅ Dataset already extracted


In [5]:
# Configuration
MODEL_PATH = local_model_path if 'local_model_path' in locals() and local_model_path else "resnet50_cars_enhanced.onnx"
MODEL_NAME = "resnet50_cars"
MODEL_TYPE = "classification"
DATASET_PATH = "stanford-cars/test" if os.path.exists("stanford-cars/test") else "stanford-cars"
CLASS_NAMES_PATH = "../class_names.json"

# Evaluation settings
EVALUATION_MODE = 'subset'
MAX_CLASSES_SUBSET = 196
IMAGES_PER_CLASS_SUBSET = 10
MAX_TOTAL_IMAGES_FULL = 0
PROGRESS_REPORT_INTERVAL = 500
DETAILED_METRICS = False

# Derived settings
MODEL_DIR = f"{MODEL_NAME}_model_artifacts"
EXPERIMENT_NAME = f"{MODEL_NAME}-{MODEL_TYPE}"
REGISTERED_MODEL_NAME = f"{MODEL_NAME}-{MODEL_TYPE}"

# Status check
model_ready = os.path.exists(MODEL_PATH)
dataset_ready = os.path.exists(DATASET_PATH)

print(f"📋 Configuration:")
print(f"   Model: {'✅' if model_ready else '❌'} {MODEL_PATH}")
print(f"   Dataset: {'✅' if dataset_ready else '❌'} {DATASET_PATH}")
print(f"   Mode: {EVALUATION_MODE} ({MAX_CLASSES_SUBSET} classes, {IMAGES_PER_CLASS_SUBSET} imgs/class)")

if model_ready and dataset_ready:
    print("🎉 Ready to proceed!")
elif not model_ready:
    print("⚠️  Model missing - evaluation will be skipped")
else:
    print("⚠️  Dataset missing - will use default metrics")

📋 Configuration:
   Model: ✅ /Users/adithyashankar/Developer/examples/edgeai/resnet50-stanford-cars/resnet50_cars_enhanced.onnx
   Dataset: ✅ stanford-cars/test
   Mode: subset (196 classes, 10 imgs/class)
🎉 Ready to proceed!


## Step 2: Environment Setup

In [6]:
# This cell removes all EdgeAI-related environment variables to ensure a clean start.
import os

edgeai_env_vars = [
    'MLFLOW_TRACKING_TOKEN', 'MLFLOW_TRACKING_URI', 
    'AWS_ACCESS_KEY_ID', 'AWS_SECRET_ACCESS_KEY', 'AWS_SESSION_TOKEN',
    'MLFLOW_S3_ENDPOINT_URL', 'MINIO_BUCKET', 'EDGEAI_SERVICE_URL',
    'EDGEAI_CATALOG_ID', 'EDGEAI_TOKEN', 'EDGEAI_EMAIL',
    'MLFLOW_REGISTRY_URI', 'MLFLOW_ARTIFACT_URI'
]

print("Clearing EdgeAI environment variables...")
cleared_count = 0
for var in edgeai_env_vars:
    if var in os.environ:
        del os.environ[var]
        cleared_count += 1

print(f"✓ Cleared {cleared_count} environment variables. Ready for fresh authentication.")

Clearing EdgeAI environment variables...
✓ Cleared 0 environment variables. Ready for fresh authentication.


In [7]:
from zededa_edgeai_sdk.client import ZededaEdgeAIClient
import getpass

print("Please enter your EdgeAI credentials:")
catalog_name = input("Enter catalog name: ")
email = input("Enter email: ")
password = getpass.getpass("Enter password: ")

client = ZededaEdgeAIClient(debug=True)
try:
    credentials = client.login(
        catalog_name,
        email=email,
        password=password 
    )
    print("\n✅ Successfully authenticated!")
except Exception as e:
    print(f"\n❌ Authentication failed: {e}")

Please enter your EdgeAI credentials:
[DEBUG] POST https://studio.edgeai.zededa.dev/api/v1/auth/login
[DEBUG] Payload: {'email': 'alice@company.com', 'password': 'pass...d123', 'catalog_id': 'external-false-test'}
[DEBUG] POST https://studio.edgeai.zededa.dev/api/v1/auth/login
[DEBUG] Payload: {'email': 'alice@company.com', 'password': 'pass...d123', 'catalog_id': 'external-false-test'}
[DEBUG] Response: 200 OK
[DEBUG] Response Body: {'access_token': 'eyJh...dRS8', 'token_type': '***', 'expires_in': 3600, 'user_info': {'user_id': 'alice', 'email': 'alice@company.com', 'full_name': 'Alice Johnson'}}
[DEBUG] GET https://studio.edgeai.zededa.dev/api/v1/user-info
[DEBUG] Headers: {'Authorization': 'Bear...dRS8'}
[DEBUG] Response: 200 OK
[DEBUG] Response Body: {'access_token': 'eyJh...dRS8', 'token_type': '***', 'expires_in': 3600, 'user_info': {'user_id': 'alice', 'email': 'alice@company.com', 'full_name': 'Alice Johnson'}}
[DEBUG] GET https://studio.edgeai.zededa.dev/api/v1/user-info
[DEB

In [8]:
# Verify environment variables
required_vars = ['MLFLOW_TRACKING_TOKEN', 'MLFLOW_TRACKING_URI', 'AWS_ACCESS_KEY_ID', 'AWS_SECRET_ACCESS_KEY', 'MLFLOW_S3_ENDPOINT_URL', 'MINIO_BUCKET']
missing = [var for var in required_vars if not os.getenv(var)]

if missing:
    print(f"❌ Missing: {', '.join(missing)}")
else:
    print("✅ All environment variables configured")

✅ All environment variables configured


## Step 4: Model Loading & Analysis

In [9]:
# Import all required libraries
import mlflow
import onnx
import onnxruntime
import os
import shutil
import random
import numpy as np
import json
import time
from pathlib import Path
from PIL import Image
from collections import defaultdict
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# PyTorch model analysis imports
import torch
from onnx2torch import convert
from torchinfo import summary
from thop import profile

print("Libraries imported successfully.")

Libraries imported successfully.


In [10]:
# Create a directory for model artifacts and copy the model file.
model_dir_path = Path(MODEL_DIR)
model_dir_path.mkdir(exist_ok=True)

model_onnx_path = Path("N/A")
onnx_model = None

if os.path.exists(MODEL_PATH):
    model_onnx_path = model_dir_path / "model.onnx"
    shutil.copy2(MODEL_PATH, model_onnx_path)
    print(f"✓ Model copied to: {model_onnx_path}")
    
    try:
        onnx_model = onnx.load(str(model_onnx_path))
        print(f"✓ ONNX model loaded successfully.")
        
        # Display input/output information
        print("\n--- Model I/O --- ")
        for i, inp in enumerate(onnx_model.graph.input):
            shape = [dim.dim_value for dim in inp.type.tensor_type.shape.dim]
            print(f"- Input {i}: \"{inp.name}\" | Shape: {shape}")
        for i, out in enumerate(onnx_model.graph.output):
            shape = [dim.dim_value for dim in out.type.tensor_type.shape.dim]
            print(f"- Output {i}: \"{out.name}\" | Shape: {shape}")
            
    except Exception as e:
        print(f"✗ Error loading ONNX model: {e}")
else:
    print(f"✗ MODEL NOT FOUND at: {MODEL_PATH}")
    print("  Please update the MODEL_PATH variable in the configuration cell.")

✓ Model copied to: resnet50_cars_model_artifacts/model.onnx
✓ ONNX model loaded successfully.

--- Model I/O --- 
- Input 0: "input" | Shape: [0, 3, 224, 224]
- Output 0: "output" | Shape: [0, 196]


### Deep Model Analysis (Optional)

The following helper function converts the ONNX model to a PyTorch representation to extract detailed information like parameter counts, FLOPs, and layer breakdowns. This provides rich metadata for the model registry.

In [11]:
def analyze_model_details(onnx_model):
    """Converts ONNX to PyTorch to extract deep model metrics."""
    model_info = {}
    if not onnx_model:
        return model_info
        
    print("\n--- Deep Model Analysis --- ")
    try:
        pytorch_model = convert(onnx_model)
        pytorch_model.eval()
    except Exception as e:
        print(f"- WARNING: Could not convert ONNX to PyTorch for analysis: {e}")
        return model_info

    # Parameters
    total_params = sum(p.numel() for p in pytorch_model.parameters())
    model_info['parameters_millions'] = total_params / 1_000_000
    print(f"- Parameters: {model_info['parameters_millions']:.2f}M")

    # FLOPs
    try:
        input_tensor = onnx_model.graph.input[0]
        input_shape = tuple(dim.dim_value if dim.dim_value > 0 else 1 for dim in input_tensor.type.tensor_type.shape.dim)
        dummy_input = torch.randn(input_shape)
        flops, _ = profile(pytorch_model, inputs=(dummy_input,), verbose=False)
        model_info['gflops'] = flops / 1_000_000_000
        print(f"- GFLOPs: {model_info['gflops']:.2f}")
    except Exception as e:
        print(f"- WARNING: Could not calculate FLOPs: {e}")

    return model_info

# Run the analysis
model_info = analyze_model_details(onnx_model)


--- Deep Model Analysis --- 
- Parameters: 24.64M
- GFLOPs: 4.09
- Parameters: 24.64M
- GFLOPs: 4.09


### Load Dataset Information

In [12]:
class_names = []
print("\n--- Dataset Information --- ")
if CLASS_NAMES_PATH and os.path.exists(CLASS_NAMES_PATH):
    try:
        with open(CLASS_NAMES_PATH, 'r') as f:
            class_names = json.load(f)
        model_info['num_classes'] = len(class_names)
        print(f"- Successfully loaded {model_info['num_classes']} class names.")
        print(f"- Sample: {class_names[:3]}...")
    except Exception as e:
        print(f"- WARNING: Error loading class names file: {e}")
else:
    print("- No class names file provided or found. Number of classes is unknown.")

# Fallback for number of classes if not in class_names.json
if 'num_classes' not in model_info and onnx_model:
    try:
        output_shape = onnx_model.graph.output[0].type.tensor_type.shape.dim
        # Find the dimension that likely corresponds to classes
        model_info['num_classes'] = [d.dim_value for d in output_shape if d.dim_value > 1][-1]
        print(f"- Inferred {model_info['num_classes']} classes from model's output shape.")
    except (IndexError, AttributeError):
        print("- Could not infer number of classes from model output.")


--- Dataset Information --- 
- Successfully loaded 196 class names.
- Sample: ['AM General Hummer SUV 2000', 'Acura Integra Type R 2001', 'Acura RL Sedan 2012']...


## Step 5: Model Performance Evaluation (Optional)

> **Developer Note:** This section evaluates the model's performance on a test dataset. The evaluation is now **fully configurable** through the parameters you set in Step 1. 

### Configuration Options:
- **EVALUATION_MODE**: Choose `'subset'` for quick validation or `'full'` for comprehensive evaluation
- **MAX_CLASSES_SUBSET**: In subset mode, limit how many classes to test (default: 50)
- **IMAGES_PER_CLASS_SUBSET**: In subset mode, how many images per class (default: 5)  
- **MAX_TOTAL_IMAGES_FULL**: In full mode, maximum total images to process (0 = no limit)
- **PROGRESS_REPORT_INTERVAL**: Print progress every N images (default: 500)
- **DETAILED_METRICS**: Enable per-class breakdown and detailed reporting (default: False)

The evaluation will automatically skip if no `DATASET_PATH` is provided and use fallback metrics instead.

In [13]:
def preprocess_image(image_path, target_size=(224, 224)):
    """Loads, resizes, and normalizes an image for ONNX inference."""
    try:
        image = Image.open(image_path).convert('RGB').resize(target_size)
        image_array = np.array(image, dtype=np.float32) / 255.0
        image_array = np.transpose(image_array, (2, 0, 1))
        image_array = np.expand_dims(image_array, axis=0)
        return image_array.astype(np.float32)
    except Exception as e:
        print(f"- WARNING: Failed to preprocess image {image_path}: {e}")
        return None

def run_evaluation(model_onnx_path, dataset_path, class_names, 
                   mode='subset', 
                   images_per_class=5, 
                   max_classes=50,
                   max_total_images=0,
                   progress_interval=500,
                   detailed_metrics=False):
    """
    Runs evaluation on the dataset with configurable parameters.
    
    Args:
        model_onnx_path: Path to the ONNX model
        dataset_path: Path to test dataset
        class_names: List of class names
        mode: 'subset' or 'full' evaluation mode
        images_per_class: For subset mode, images per class to test
        max_classes: For subset mode, maximum classes to test
        max_total_images: For full mode, maximum total images (0 = no limit)
        progress_interval: Print progress every N images
        detailed_metrics: Include per-class metrics and confusion matrix
    """
    print(f"\n--- Running Model Evaluation (Mode: {mode.upper()}) ---")
    if not os.path.exists(dataset_path):
        print(f"- WARNING: Dataset not found at '{dataset_path}'. Skipping evaluation.")
        return None
        
    # Setup ONNX inference session
    try:
        ort_session = onnxruntime.InferenceSession(str(model_onnx_path))
        input_name = ort_session.get_inputs()[0].name
        output_name = ort_session.get_outputs()[0].name
        print(f"- ONNX session initialized successfully")
    except Exception as e:
        print(f"- ERROR: Failed to initialize ONNX session: {e}")
        return None
    
    # Collect test data
    class_dirs = sorted([d for d in os.listdir(dataset_path) if os.path.isdir(os.path.join(dataset_path, d))])
    if not class_dirs:
        print("- ERROR: No class directories found in the dataset path.")
        return None
    
    original_class_count = len(class_dirs)
    
    # Apply mode-specific filtering
    if mode == 'subset':
        class_dirs = class_dirs[:max_classes]
        print(f"- Subset mode: Testing {len(class_dirs)} out of {original_class_count} classes")
    else:
        print(f"- Full mode: Testing all {len(class_dirs)} classes")

    class_to_idx = {name: i for i, name in enumerate(class_dirs)}
    
    # Collect images with robust handling
    test_data = []
    images_collected_per_class = {}
    classes_with_insufficient_images = []
    total_available_images = 0
    
    for class_name in class_dirs:
        class_path = os.path.join(dataset_path, class_name)
        image_files = [os.path.join(class_path, f) for f in os.listdir(class_path) 
                      if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff'))]
        random.shuffle(image_files)
        
        available_images = len(image_files)
        total_available_images += available_images
        
        if mode == 'subset':
            # Take minimum of requested images and available images
            images_to_take = min(images_per_class, available_images)
            image_files = image_files[:images_to_take]
            
            # Track classes with insufficient images for user feedback
            if available_images < images_per_class:
                classes_with_insufficient_images.append((class_name, available_images, images_per_class))
                
        elif mode == 'full' and max_total_images > 0:
            # For full mode with limit, distribute images evenly across classes
            images_per_class_full = max_total_images // len(class_dirs)
            images_to_take = min(images_per_class_full, available_images)
            image_files = image_files[:images_to_take]
            
        images_collected_per_class[class_name] = len(image_files)
        
        for img_file in image_files:
            test_data.append((img_file, class_to_idx[class_name]))

    # Provide detailed feedback about image collection
    print(f"- Collected {len(test_data)} images across {len(class_dirs)} classes")
    
    if mode == 'subset':
        avg_images = np.mean(list(images_collected_per_class.values()))
        min_images = min(images_collected_per_class.values())
        max_images = max(images_collected_per_class.values())
        print(f"- Images per class: avg={avg_images:.1f}, min={min_images}, max={max_images}")
        
        # Warn about classes with insufficient images
        if classes_with_insufficient_images:
            print(f"- WARNING: {len(classes_with_insufficient_images)} classes have fewer than {images_per_class} requested images:")
            # Show up to 5 examples
            for i, (class_name, available, requested) in enumerate(classes_with_insufficient_images[:5]):
                print(f"    {class_name}: {available}/{requested} images")
            if len(classes_with_insufficient_images) > 5:
                print(f"    ... and {len(classes_with_insufficient_images) - 5} more classes")
            print(f"- Note: This is normal for datasets with variable class sizes. Using all available images.")
    
    # Additional validation
    if len(test_data) == 0:
        print("- ERROR: No images collected for evaluation.")
        return None
    
    # Run inference with progress tracking
    predictions, true_labels, inference_times = [], [], []
    failed_predictions = 0
    
    print("- Starting inference...")
    start_time_total = time.time()
    
    for i, (img_path, true_label) in enumerate(test_data):
        if i > 0 and i % progress_interval == 0: 
            elapsed = time.time() - start_time_total
            avg_time = elapsed / i
            remaining = (len(test_data) - i) * avg_time
            print(f"  ...processed {i}/{len(test_data)} images ({i/len(test_data)*100:.1f}%) | "
                  f"ETA: {remaining/60:.1f} min")
            
        preprocessed = preprocess_image(img_path)
        if preprocessed is None: 
            failed_predictions += 1
            continue
        
        start_time = time.time()
        try:
            outputs = ort_session.run([output_name], {input_name: preprocessed})
            inference_time = (time.time() - start_time) * 1000
            inference_times.append(inference_time)
            predictions.append(np.argmax(outputs[0][0]))
            true_labels.append(true_label)
        except Exception as e:
            print(f"  WARNING: Inference failed for {img_path}: {e}")
            failed_predictions += 1
            continue

    total_time = time.time() - start_time_total
    
    if not predictions:
        print("- ERROR: No successful predictions were made.")
        return None

    # Calculate basic metrics
    accuracy = accuracy_score(true_labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(true_labels, predictions, average='weighted', zero_division=0)
    avg_inference_time = np.mean(inference_times)

    print(f"\n--- Evaluation Results ---")
    print(f"- Total Images Processed: {len(predictions)} (Failed: {failed_predictions})")
    print(f"- Total Evaluation Time: {total_time/60:.2f} minutes")
    print(f"- Accuracy: {accuracy:.2%}")
    print(f"- Precision: {precision:.3f}")
    print(f"- Recall: {recall:.3f}")
    print(f"- F1-Score: {f1:.3f}")
    print(f"- Avg Inference Time: {avg_inference_time:.2f} ms")
    print(f"- Throughput: {len(predictions)/total_time:.1f} images/sec")
    
    results = {
        'accuracy': accuracy, 
        'precision': precision, 
        'recall': recall, 
        'f1_score': f1, 
        'inference_time_ms': avg_inference_time,
        'total_images': len(predictions),
        'failed_images': failed_predictions,
        'evaluation_time_minutes': total_time/60,
        'throughput_imgs_per_sec': len(predictions)/total_time,
        'classes_evaluated': len(class_dirs),
        'avg_images_per_class': np.mean(list(images_collected_per_class.values()))
    }
    
    # Add detailed metrics if requested
    if detailed_metrics:
        from sklearn.metrics import classification_report, confusion_matrix
        print(f"\n--- Detailed Metrics ---")
        
        # Per-class metrics
        class_names_subset = [class_dirs[i] for i in range(len(class_dirs))]
        report = classification_report(true_labels, predictions, 
                                     target_names=class_names_subset, 
                                     output_dict=True, zero_division=0)
        
        print("Per-class Performance (Top 10 best accuracy):")
        class_accuracies = [(name, metrics['precision']) for name, metrics in report.items() 
                           if isinstance(metrics, dict) and 'precision' in metrics]
        class_accuracies.sort(key=lambda x: x[1], reverse=True)
        
        for i, (class_name, acc) in enumerate(class_accuracies[:10]):
            print(f"  {i+1:2d}. {class_name}: {acc:.3f}")
            
        results['detailed_report'] = report
    
    return results

# --- Run Evaluation with User Configuration --- #
print("=" * 60)
print("STARTING MODEL EVALUATION")
print("=" * 60)

evaluation_results = {}
if DATASET_PATH and os.path.exists(DATASET_PATH):
    print(f"Using configuration:")
    print(f"  - Mode: {EVALUATION_MODE}")
    print(f"  - Dataset: {DATASET_PATH}")
    
    if EVALUATION_MODE == 'subset':
        print(f"  - Max Classes: {MAX_CLASSES_SUBSET}")
        print(f"  - Images per Class (requested): {IMAGES_PER_CLASS_SUBSET}")
        print(f"  - Note: Will use fewer images if class doesn't have enough")
    
    # Run evaluation with user-configured parameters
    evaluation_results = run_evaluation(
        model_onnx_path=model_onnx_path, 
        dataset_path=DATASET_PATH, 
        class_names=class_names, 
        mode=EVALUATION_MODE,
        images_per_class=IMAGES_PER_CLASS_SUBSET,
        max_classes=MAX_CLASSES_SUBSET,
        max_total_images=MAX_TOTAL_IMAGES_FULL,
        progress_interval=PROGRESS_REPORT_INTERVAL,
        detailed_metrics=DETAILED_METRICS
    )
    
else:
    print(f"Dataset path not provided or doesn't exist: {DATASET_PATH}")
    print("Skipping evaluation and using default metrics.")
    
if not evaluation_results:
    print("\n- Using default performance metrics for upload.")
    # Fallback metrics if evaluation is skipped or fails
    evaluation_results = {
        'accuracy': 0.873, 
        'precision': 0.873, 
        'recall': 0.873, 
        'f1_score': 0.873, 
        'inference_time_ms': 45.0,
        'total_images': 0,
        'failed_images': 0,
        'evaluation_time_minutes': 0,
        'throughput_imgs_per_sec': 0,
        'classes_evaluated': 0,
        'avg_images_per_class': 0
    }

print(f"\n{'='*60}")
print("EVALUATION COMPLETE - Updating model info...")
print(f"{'='*60}")

# Update model_info with evaluation results
model_info.update(evaluation_results)

STARTING MODEL EVALUATION
Using configuration:
  - Mode: subset
  - Dataset: stanford-cars/test
  - Max Classes: 196
  - Images per Class (requested): 10
  - Note: Will use fewer images if class doesn't have enough

--- Running Model Evaluation (Mode: SUBSET) ---
- ONNX session initialized successfully
- Subset mode: Testing 196 out of 196 classes
- Collected 1960 images across 196 classes
- Images per class: avg=10.0, min=10, max=10
- Starting inference...
- Collected 1960 images across 196 classes
- Images per class: avg=10.0, min=10, max=10
- Starting inference...
  ...processed 500/1960 images (25.5%) | ETA: 0.9 min
  ...processed 500/1960 images (25.5%) | ETA: 0.9 min
  ...processed 1000/1960 images (51.0%) | ETA: 0.6 min
  ...processed 1000/1960 images (51.0%) | ETA: 0.6 min
  ...processed 1500/1960 images (76.5%) | ETA: 0.3 min
  ...processed 1500/1960 images (76.5%) | ETA: 0.3 min

--- Evaluation Results ---
- Total Images Processed: 1960 (Failed: 0)
- Total Evaluation Time: 1.

## Step 6: Configuration Generation

> **Developer Note:** This cell automatically generates the `config.pbtxt` required by Triton/OpenVino Inference Server. It dynamically pulls information from your loaded ONNX model, so you don't need to change anything here.

In [14]:
def generate_triton_config(model_name, onnx_model):
    """Dynamically generates a Triton config.pbtxt from an ONNX model."""
    config = {
        'name': model_name,
        'backend': 'onnxruntime',
        'max_batch_size': 32,
        'input': [],
        'output': []
    }

    # Data type mapping from ONNX to Triton
    dtype_map = {
        1: 'TYPE_FP32', 2: 'TYPE_UINT8', 3: 'TYPE_INT8', 4: 'TYPE_UINT16',
        5: 'TYPE_INT16', 6: 'TYPE_INT32', 7: 'TYPE_INT64', 9: 'TYPE_BOOL',
        10: 'TYPE_FP16', 11: 'TYPE_FP64', 12: 'TYPE_UINT32', 13: 'TYPE_UINT64'
    }
    
    # Process inputs
    for inp in onnx_model.graph.input:
        dims = [d.dim_value for d in inp.type.tensor_type.shape.dim][1:] # Exclude batch dim
        config['input'].append({
            'name': inp.name,
            'data_type': dtype_map.get(inp.type.tensor_type.elem_type, 'TYPE_INVALID'),
            'dims': dims
        })
    
    # Process outputs
    for out in onnx_model.graph.output:
        dims = [d.dim_value for d in out.type.tensor_type.shape.dim][1:] # Exclude batch dim
        config['output'].append({
            'name': out.name,
            'data_type': dtype_map.get(out.type.tensor_type.elem_type, 'TYPE_INVALID'),
            'dims': dims
        })

    # Build the .pbtxt string
    pbtxt = f'name: "{config["name"]}"\nbackend: "{config["backend"]}"\nmax_batch_size: {config["max_batch_size"]}\n'
    for io_type in ['input', 'output']:
        for item in config[io_type]:
            pbtxt += f'\n{io_type} [{{ \
  name: "{item["name"]}" \
  data_type: {item["data_type"]} \
  dims: {str(item["dims"])} \
}}]'
    
    return pbtxt

if onnx_model:
    print("--- Generating Deployment Configurations ---")
    # Generate and save Triton config
    config_content = generate_triton_config(MODEL_NAME, onnx_model)
    config_path = model_dir_path / "config.pbtxt"
    config_path.write_text(config_content)
    print(f"- ✓ Triton config generated: {config_path}")

    # Generate and save a simple requirements.txt
    requirements_content = "onnxruntime-gpu>=1.13.0\nnumpy>=1.21.0\npillow>=8.0.0"
    requirements_path = model_dir_path / "requirements.txt"
    requirements_path.write_text(requirements_content)
    print(f"- ✓ Requirements.txt generated: {requirements_path}")
else:
    print("- WARNING: ONNX model not loaded. Skipping config generation.")

--- Generating Deployment Configurations ---
- ✓ Triton config generated: resnet50_cars_model_artifacts/config.pbtxt
- ✓ Requirements.txt generated: resnet50_cars_model_artifacts/requirements.txt


## Step 7: Model Upload

In [15]:
# Create or set the MLflow experiment.
try:
    mlflow.set_experiment(EXPERIMENT_NAME)
    print(f"✓ MLflow experiment set to: '{EXPERIMENT_NAME}'")
except Exception as e:
    print(f"❌ Could not set MLflow experiment: {e}")

2025/10/23 21:09:46 INFO mlflow.tracking.fluent: Experiment with name 'resnet50_cars-classification' does not exist. Creating a new experiment.


✓ MLflow experiment set to: 'resnet50_cars-classification'


In [16]:
# Start an MLflow run to log the model, artifacts, parameters, and metrics.
run_id = None
if onnx_model:
    with mlflow.start_run(run_name=f"{MODEL_NAME}-upload") as run:
        run_id = run.info.run_id
        print(f"--- Starting MLflow Run: {run_id} ---")
        
        # Log parameters and metrics from the model_info dictionary
        print("- Logging parameters and metrics...")
        mlflow.log_param("model_name", MODEL_NAME)
        mlflow.log_param("model_type", MODEL_TYPE)
        for key, value in model_info.items():
            if isinstance(value, (int, float)):
                mlflow.log_metric(key, value)
            else:
                mlflow.log_param(key, value)

        # Set descriptive tags
        mlflow.set_tags({
            "model_name": MODEL_NAME,
            "deployment_ready": "true",
            "framework": "ONNX",
            "task": MODEL_TYPE
        })
        
        # Log the ONNX model and register it
        print(f"- Logging and registering model as '{REGISTERED_MODEL_NAME}'...")
        
        # Show model file size
        if model_onnx_path.exists():
            model_size_mb = model_onnx_path.stat().st_size / (1024 * 1024)
            print(f"  Model file size: {model_size_mb:.2f} MB")
        
        mlflow.onnx.log_model(
            onnx_model=onnx_model,
            artifact_path="model",
            registered_model_name=REGISTERED_MODEL_NAME,
            save_as_external_data=False
        )

        # Log additional artifacts
        print("- Logging artifacts (config.pbtxt, etc.)...")
        
        # Show artifact sizes
        if model_dir_path.exists():
            total_artifact_size = 0
            artifact_files = []
            for file_path in model_dir_path.rglob('*'):
                if file_path.is_file():
                    size_bytes = file_path.stat().st_size
                    total_artifact_size += size_bytes
                    artifact_files.append((file_path.name, size_bytes))
            
            print(f"  Artifact files:")
            for file_name, size_bytes in sorted(artifact_files, key=lambda x: x[1], reverse=True):
                size_kb = size_bytes / 1024
                if size_kb < 1024:
                    print(f"    - {file_name}: {size_kb:.2f} KB")
                else:
                    print(f"    - {file_name}: {size_kb/1024:.2f} MB")
            print(f"  Total artifacts size: {total_artifact_size / (1024 * 1024):.2f} MB")
        
        mlflow.log_artifacts(MODEL_DIR, artifact_path="model-artifacts")
        
        print(f"\n✅ MLflow run completed successfully.")
else:
    print("❌ ONNX model not loaded. Skipping MLflow run.")

--- Starting MLflow Run: 046adc69e4cb446b8f47788a212d0e35 ---
- Logging parameters and metrics...


2025/10/23 21:09:52 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


- Logging and registering model as 'resnet50_cars-classification'...
  Model file size: 94.00 MB


2025/10/23 21:09:56 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
Successfully registered model 'resnet50_cars-classification'.
2025/10/23 21:10:04 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: resnet50_cars-classification, version 1
Successfully registered model 'resnet50_cars-classification'.
2025/10/23 21:10:04 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: resnet50_cars-classification, version 1
Created version '1' of model 'resnet50_cars-classification'.
Created version '1' of model 'resnet50_cars-classification'.


- Logging artifacts (config.pbtxt, etc.)...
  Artifact files:
    - model.onnx: 94.00 MB
    - config.pbtxt: 0.20 KB
    - requirements.txt: 0.05 KB
  Total artifacts size: 94.00 MB

✅ MLflow run completed successfully.
🏃 View run resnet50_cars-upload at: https://studio.edgeai.zededa.dev/#/experiments/1/runs/046adc69e4cb446b8f47788a212d0e35
🧪 View experiment at: https://studio.edgeai.zededa.dev/#/experiments/1

✅ MLflow run completed successfully.
🏃 View run resnet50_cars-upload at: https://studio.edgeai.zededa.dev/#/experiments/1/runs/046adc69e4cb446b8f47788a212d0e35
🧪 View experiment at: https://studio.edgeai.zededa.dev/#/experiments/1


## Step 8: Registration & Deployment

In [17]:
# Transition the newly registered model version to the "Production" stage.
from mlflow.tracking import MlflowClient

client = MlflowClient()
model_version = None

print(f"--- Transitioning Model '{REGISTERED_MODEL_NAME}' to Production ---")
try:
    latest_versions = client.get_latest_versions(REGISTERED_MODEL_NAME, stages=["None"])
    if latest_versions:
        model_version = latest_versions[0]
        
        # Add a description to the model version
        acc = model_info.get('accuracy', 0)
        inf_time = model_info.get('inference_time_ms', 0)
        desc = f"{MODEL_NAME} ({MODEL_TYPE}) with {acc:.1%} accuracy and {inf_time:.1f}ms inference time."
        client.update_model_version(name=REGISTERED_MODEL_NAME, version=model_version.version, description=desc)
        
        # Transition to Production
        client.transition_model_version_stage(
            name=REGISTERED_MODEL_NAME,
            version=model_version.version,
            stage="Production"
        )
        print(f"✓ Successfully transitioned version {model_version.version} to 'Production'.")
    else:
        print("- WARNING: No new model version found in 'None' stage to transition.")
except Exception as e:
    print(f"❌ Error during model transition: {e}")

--- Transitioning Model 'resnet50_cars-classification' to Production ---


/var/folders/j7/0fwb424x58l7_mz28_fv0nlh0000gn/T/ipykernel_73885/1004548785.py:9: FutureWarning: ``mlflow.tracking.client.MlflowClient.get_latest_versions`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  latest_versions = client.get_latest_versions(REGISTERED_MODEL_NAME, stages=["None"])


✓ Successfully transitioned version 1 to 'Production'.


/var/folders/j7/0fwb424x58l7_mz28_fv0nlh0000gn/T/ipykernel_73885/1004548785.py:20: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


## Step 9: Verification

In [18]:
# Verify the final status of the uploaded model.
print("--- Final Verification Summary ---")

if run_id and model_version:
    print(f"✅ Success!")
    print(f"  - Run ID: {run_id}")
    print(f"  - Model Name: {model_version.name}")
    print(f"  - Model Version: {model_version.version}")
    print(f"  - Current Stage: {model_version.current_stage}")
    
    accuracy = model_info.get('accuracy', 0)
    inference_time = model_info.get('inference_time_ms', 0)
    print(f"\n  - Accuracy: {accuracy:.2%}")
    print(f"  - Inference Time: {inference_time:.2f} ms")
    
    print("\n🚀 Model is uploaded, registered, and ready for deployment from the EdgeAI platform.")
else:
    print("❌ Verification failed. Please review the output of the previous cells for errors.")

--- Final Verification Summary ---
✅ Success!
  - Run ID: 046adc69e4cb446b8f47788a212d0e35
  - Model Name: resnet50_cars-classification
  - Model Version: 1
  - Current Stage: None

  - Accuracy: 80.20%
  - Inference Time: 29.55 ms

🚀 Model is uploaded, registered, and ready for deployment from the EdgeAI platform.
